# Reasoning Agent: Natural Language KB → Logic → Classical Reasoning → True/False

**Pipeline**

`context + statement  →  rule-based parser  →  facts / rules / causal equations / FOL clauses  →  classical reasoning  →  True / False`

| Input kind | Representation | Reasoning algorithm |
|---|---|---|
| RuleTaker style English rules | literals `pred(args)` + Horn-like rules with variables | forward chaining with unification, stratified negation-as-failure (closed world) |
| CounterBench causal stories | boolean structural equations over a causal graph | abduction by enumeration (model checking) + intervention + forward propagation in topological order |
| FOLIO with FOL annotation | clauses in CNF after Skolemisation | resolution refutation with unification, set-of-support + unit preference |
| FOLIO / other free English (no annotation) | our own English → FOL translation (quantifier, negation, if-then, either-or, neither-nor patterns) | same resolution prover, open world |

No generative model is used anywhere; every answer comes from the engines below.

## 0. Setup
Change `DATA_DIR` / `TEST_FILE` here.

In [1]:
import os
import re
import json
import time
import random
from collections import Counter, defaultdict

import pandas as pd

# Folder that holds the downloaded public datasets (see README.txt).
# The public data is only needed for the evaluation sections; predictions for
# TEST_FILE work without it.
DATA_DIR = next((d for d in [os.path.join("..", "data"), "data"] if os.path.isdir(d)), os.path.join("..", "data"))

# File used to produce submission.csv. Point this to the private test file.
# Accepted formats: .csv / .json / .jsonl with a context column and a statement
# column (several common column names are recognised, see load_any_file).
TEST_FILE = None          # e.g. "private_test.csv"; None -> public evaluation set
SUBMISSION_PATH = "submission.csv"

# Semantics used for rule-style knowledge bases.
# True  = closed-world assumption: a negated literal holds when its positive form
#         cannot be derived (negation as failure). RuleTaker is built this way;
#         on our 10,000-query sample it gives 99.94% vs 75.11% for open world.
# False = open world: "not P" holds only when it is stated or derived explicitly.
CLOSED_WORLD = True

random.seed(0)

## 1. Data
All three datasets are mapped to one table: `id, context, statement, label (True/False), source, subset`.

In [2]:
def read_jsonl(path):
    with open(path, encoding="utf-8") as fh:
        return [json.loads(line) for line in fh if line.strip()]


def load_ruletaker(split="test"):
    rows = read_jsonl(os.path.join(DATA_DIR, f"ruletaker_{split}.jsonl"))
    return pd.DataFrame({
        "id": [f"ruletaker_{split}_{i}" for i in range(len(rows))],
        "context": [r["context"] for r in rows],
        "statement": [r["question"] for r in rows],
        "label": [r["label"] == "entailment" for r in rows],
        "source": "ruletaker",
        "subset": [r["config"] for r in rows],
    })


def load_folio(split="validation"):
    rows = read_jsonl(os.path.join(DATA_DIR, f"folio_{split}.jsonl"))
    return pd.DataFrame({
        "id": [f"folio_{split}_{r['example_id']}" for r in rows],
        "context": [r["premises"] for r in rows],
        "statement": [r["conclusion"] for r in rows],
        # 'Uncertain' and 'False' both mean the conclusion is NOT entailed
        "label": [r["label"] == "True" for r in rows],
        "source": "folio",
        "subset": [r["label"] for r in rows],
        "context_fol": [r["premises-FOL"] for r in rows],
        "statement_fol": [r["conclusion-FOL"] for r in rows],
    })


def load_counterbench():
    frames = []
    for name in ["data_balanced_alpha_V1.json", "data_balanced_backdoor_V2.json"]:
        with open(os.path.join(DATA_DIR, name), encoding="utf-8") as fh:
            items = json.load(fh)
        tag = name.split("_")[2]
        frames.append(pd.DataFrame({
            "id": [f"counterbench_{tag}_{x['question_id']}_{i}" for i, x in enumerate(items)],
            "context": [x["given_info"] for x in items],
            "statement": [x["question"] for x in items],
            "label": [x["answer"].strip().lower() == "yes" for x in items],
            "source": "counterbench",
            "subset": [x["type"] for x in items],
        }))
    return pd.concat(frames, ignore_index=True)


try:
    ruletaker_test = load_ruletaker("test")
    folio_train, folio_val = load_folio("train"), load_folio("validation")
    counterbench = load_counterbench()
    PUBLIC_DATA_AVAILABLE = True
except FileNotFoundError as missing:
    # e.g. only the private test file is present: skip the public evaluation
    print(f"Public datasets not found ({missing.filename}); evaluation sections are skipped.")
    PUBLIC_DATA_AVAILABLE = False

for name, df in [] if not PUBLIC_DATA_AVAILABLE else [("RuleTaker test", ruletaker_test), ("FOLIO train", folio_train),
                 ("FOLIO validation", folio_val), ("CounterBench", counterbench)]:
    print(f"{name:18s} {len(df):7d} examples   share of True labels = {df['label'].mean():.3f}")

RuleTaker test      151911 examples   share of True labels = 0.503
FOLIO train           1001 examples   share of True labels = 0.388
FOLIO validation       203 examples   share of True labels = 0.355
CounterBench          1200 examples   share of True labels = 0.496


## 2. Representation and natural-language parser (RuleTaker style)
A **literal** is `Literal(pred, args, negated)`; variables start with `?`. A **rule** is a list of condition literals plus one conclusion literal.
The parser uses sentence templates: facts (`X is (not) A`, `X (does not) verb Y`), conditional rules (`If ... then ...`, also without `then`), and group rules (`All red, round things are kind`, `Big people are rough`, `Someone who is A is B`).

In [3]:
import re
from collections import namedtuple, defaultdict

# A literal is a predicate applied to arguments, possibly negated.
#   young(bob)            -> Literal('young', ('bob',), False)
#   not chase(cat, cow)   -> Literal('chase', ('cat', 'cow'), True)
# Arguments starting with '?' are variables.
Literal = namedtuple("Literal", ["pred", "args", "negated"])
Rule = namedtuple("Rule", ["conditions", "conclusion", "text"])

VARIABLE = "?x"
PRONOUNS = {"someone", "something", "they", "it", "he", "she", "somebody", "anyone",
            "anything", "everyone", "everything", "person", "thing", "people", "things"}


def is_variable(term):
    return term.startswith("?")


def literal_to_str(lit):
    body = f"{lit.pred}({', '.join(lit.args)})"
    return ("¬" + body) if lit.negated else body

In [4]:
# ----------------------------------------------------------------------------
# Small helpers for normalising words
# ----------------------------------------------------------------------------
def verb_base_form(verb):
    """chases -> chase, sees -> see, carries -> carry, does -> do."""
    if verb in ("is", "are", "was", "were"):
        return "be"
    if verb in ("does", "do"):
        return "do"
    if verb in ("has", "have"):
        return "have"
    if verb.endswith("ies") and len(verb) > 4:
        return verb[:-3] + "y"
    if verb.endswith(("sses", "ches", "shes", "xes", "zes")):
        return verb[:-2]
    if verb.endswith("s") and not verb.endswith("ss"):
        return verb[:-1]
    return verb


def entity_name(phrase):
    """'the bald eagle' -> 'bald_eagle', 'Bob' -> 'bob', pronouns -> variable."""
    words = phrase.strip().lower().split()
    if words and words[0] in ("the", "a", "an"):
        words = words[1:]
    if not words:
        return None
    if len(words) == 1 and words[0] in PRONOUNS:
        return VARIABLE
    return "_".join(words)

In [5]:
# ----------------------------------------------------------------------------
# Clause parser:  "<subject> is (not) <adj>"  /  "<subject> (does not) <verb> <object>"
# ----------------------------------------------------------------------------
NOUN_PHRASE = r"(?:the |a |an )?[a-z]+(?: [a-z]+)?"
ADJ_CLAUSE = re.compile(rf"^({NOUN_PHRASE}?) (?:is|are) (not )?([a-z]+)$")
VERB_CLAUSE = re.compile(rf"^({NOUN_PHRASE}?) (does not |do not |doesn't |don't )?([a-z]+) ((?:the |a |an )[a-z]+(?: [a-z]+)?|[a-z]+)$")


def parse_clause(text, default_subject=None):
    """Parse one simple clause into a Literal. Returns None if it does not fit.

    default_subject is used when the clause has no subject of its own,
    e.g. the 'smart' in 'if someone is young and smart'.
    """
    text = text.strip()
    match = ADJ_CLAUSE.match(text)
    if match:
        subject = entity_name(match.group(1))
        if subject is None or subject.split("_")[-1] in ("is", "are"):
            return None
        return Literal(match.group(3), (subject,), bool(match.group(2)))

    match = VERB_CLAUSE.match(text)
    if match:
        subject_phrase, negation, verb, obj = match.groups()
        subject = entity_name(subject_phrase)
        if verb in ("is", "are", "not") or subject is None:
            return None
        return Literal(verb_base_form(verb), (subject, entity_name(obj)), bool(negation))

    # Clauses that only carry a predicate, subject comes from the previous clause.
    if default_subject is not None:
        words = text.split()
        negated = False
        if words[:1] == ["not"]:
            negated, words = True, words[1:]
        if words[:2] in (["does", "not"], ["do", "not"]):
            negated, words = True, words[2:]
        if words[:1] in (["is"], ["are"]):
            words = words[1:]
            if words[:1] == ["not"]:
                negated, words = True, words[1:]
        if len(words) == 1:
            return Literal(words[0], (default_subject,), negated)
        if len(words) >= 2 and words[1] in ("the", "a", "an") or len(words) == 2:
            return Literal(verb_base_form(words[0]), (default_subject, entity_name(" ".join(words[1:]))), negated)
    return None


def parse_conjunction(text):
    """'someone is young and not kind and they like the cat' -> list of Literals."""
    literals = []
    previous_subject = None
    for part in re.split(r",? and |, ", text):
        lit = parse_clause(part, default_subject=previous_subject)
        if lit is None:
            return None
        literals.append(lit)
        previous_subject = lit.args[0]
    return literals

In [6]:
# ----------------------------------------------------------------------------
# Sentence parser: decides whether a sentence is a fact or a rule
# ----------------------------------------------------------------------------
GROUP_WORDS = r"(?:things|people|ones|animals|persons|someone|something)"
ALL_RULE = re.compile(rf"^(?:all |every )?([a-z, ]+?) {GROUP_WORDS} (?:are|is) (.+)$")
WHO_RULE = re.compile(r"^(?:someone|something|anyone|anything|people|things|everyone|everything) (?:who|that) (?:is|are) (.+?) (?:is|are) (.+)$")


def make_rule(conditions, conclusions, text):
    return [Rule(tuple(conditions), head, text) for head in conclusions]


def parse_sentence(sentence):
    """Returns ('fact', [Literal]) or ('rule', [Rule]) or (None, None)."""
    text = sentence.strip().lower().rstrip(".").strip()
    text = re.sub(r"\s+", " ", text)
    text = re.sub(r",\s*then ", " then ", text)
    if not text:
        return None, None

    # "If <conditions> then <conclusions>"  (the 'then' is sometimes missing)
    if text.startswith("if "):
        body = text[3:]
        candidates = []
        if " then " in body:
            cond_text, concl_text = body.split(" then ", 1)
            candidates.append((cond_text, concl_text))
        else:
            # try every split point "... they ..." / "... it ..."
            for m in re.finditer(r" (?=(?:they|it|he|she|the |[a-z]+ (?:is|are|does) ))", body):
                candidates.append((body[:m.start()], body[m.end():]))
        for cond_text, concl_text in candidates:
            conditions = parse_conjunction(cond_text)
            conclusions = parse_conjunction(concl_text)
            if conditions and conclusions:
                return "rule", make_rule(conditions, conclusions, sentence.strip())
        return None, None

    # "Someone who is X is Y"
    match = WHO_RULE.match(text)
    if match:
        conditions = parse_conjunction("someone is " + match.group(1))
        conclusions = parse_conjunction("someone is " + match.group(2))
        if conditions and conclusions:
            return "rule", make_rule(conditions, conclusions, sentence.strip())

    # "All red, round things are (not) kind"   /   "Big people are rough"
    match = ALL_RULE.match(text)
    if match and not text.startswith("the "):
        adjectives = [a for a in re.split(r",\s*|\s+and\s+|\s+", match.group(1)) if a]
        conditions = [Literal(a, (VARIABLE,), False) for a in adjectives]
        conclusions = parse_conjunction("someone is " + match.group(2))
        if conditions and conclusions:
            return "rule", make_rule(conditions, conclusions, sentence.strip())

    # Otherwise it must be a fact (possibly a conjunction: "Bob is big and red")
    literals = parse_conjunction(text)
    if literals and all(not is_variable(a) for lit in literals for a in lit.args):
        return "fact", literals
    return None, None


def split_sentences(text):
    return [s for s in re.split(r"(?<=[.!?])\s+|\n+", text.strip()) if s.strip()]


class KnowledgeBase:
    def __init__(self):
        self.facts = set()
        self.rules = []
        self.unparsed = []
        self.sentence_count = 0

    @classmethod
    def from_text(cls, context):
        kb = cls()
        for sentence in split_sentences(context):
            kb.sentence_count += 1
            kind, items = parse_sentence(sentence)
            if kind == "fact":
                kb.facts.update(items)
            elif kind == "rule":
                kb.rules.extend(items)
            else:
                kb.unparsed.append(sentence)
        return kb

    def parsed_fraction(self):
        if self.sentence_count == 0:
            return 0.0
        return 1 - len(self.unparsed) / self.sentence_count

    def constants(self):
        names = set()
        for lit in self.facts:
            names.update(lit.args)
        for rule in self.rules:
            for lit in rule.conditions + (rule.conclusion,):
                names.update(a for a in lit.args if not is_variable(a))
        return names

## 3. Forward chaining with unification
* Conditions are matched against the fact base by depth-first search with unification.
* `not P` in a condition is negation as failure. Rules are split into **strata** so that a rule testing `not P` only runs after every rule that can derive `P` has reached its fixed point.
* For a positive query the engine stops as soon as the query is derived (early stopping).
* For every derived fact we store the rule and premises that produced it, which gives the proof trace.

In [7]:
# ----------------------------------------------------------------------------
# Unification and forward chaining
# ----------------------------------------------------------------------------
def unify_args(pattern_args, fact_args, bindings):
    """Unify a tuple of (possibly variable) terms with ground terms."""
    new_bindings = dict(bindings)
    for p, f in zip(pattern_args, fact_args):
        if is_variable(p):
            if p in new_bindings and new_bindings[p] != f:
                return None
            new_bindings[p] = f
        elif p != f:
            return None
    return new_bindings


def substitute(lit, bindings):
    return Literal(lit.pred, tuple(bindings.get(a, a) for a in lit.args), lit.negated)


class ForwardChainer:
    """Derives every fact that follows from the KB.

    closed_world=True : 'not X' in a rule condition holds when X cannot be derived
                        (negation as failure, evaluated stratum by stratum).
    closed_world=False: 'not X' only holds if the negative literal itself is known.
    """

    def __init__(self, kb, closed_world=True, max_rounds=100, split_by_sign=True):
        self.kb = kb
        self.closed_world = closed_world
        self.split_by_sign = split_by_sign      # False = layer rules by predicate name only (ablation)
        self.max_rounds = max_rounds
        self.derived_from = {}          # fact -> (rule, [premise facts])  used for traces
        self.facts_by_pred = defaultdict(set)
        self.known = set()
        self.domain = sorted(kb.constants())
        for fact in kb.facts:
            self._add(fact, None)

    def _add(self, fact, reason):
        if fact in self.known:
            return False
        self.known.add(fact)
        self.facts_by_pred[(fact.pred, fact.negated)].add(fact.args)
        self.derived_from[fact] = reason
        return True

    def holds(self, lit):
        """Truth of a ground literal given what is currently known."""
        if not lit.negated:
            return lit in self.known
        if lit in self.known:
            return True
        if self.closed_world:
            return Literal(lit.pred, lit.args, False) not in self.known
        return False

    def _match(self, conditions, bindings, used):
        """Depth-first search over condition list, yields (bindings, premises)."""
        if not conditions:
            yield bindings, used
            return
        first, rest = conditions[0], conditions[1:]
        unbound = [a for a in first.args if is_variable(a) and a not in bindings]
        if not unbound:
            ground = substitute(first, bindings)
            if self.holds(ground):
                yield from self._match(rest, bindings, used + [ground])
            return
        if first.negated and self.closed_world:
            # NAF condition with a free variable: try every constant in the domain
            for const in self.domain:
                b = dict(bindings)
                for var in unbound:
                    b[var] = const
                ground = substitute(first, b)
                if self.holds(ground):
                    yield from self._match(rest, b, used + [ground])
            return
        for args in list(self.facts_by_pred.get((first.pred, first.negated), ())):
            b = unify_args(first.args, args, bindings)
            if b is not None:
                yield from self._match(rest, b, used + [Literal(first.pred, args, first.negated)])

    def _strata(self):
        """Group rules so that a rule using 'not P' runs only after every rule
        that can produce P has finished (stratified negation as failure).

        Levels are kept per (predicate, sign): a rule concluding 'not P' does
        not have to wait for itself, only rules that test 'not P' depend on P.
        """
        if not self.closed_world:
            return [self.kb.rules]
        level = defaultdict(int)
        sign = (lambda negated: negated) if self.split_by_sign else (lambda negated: False)
        head_keys = {(r.conclusion.pred, sign(r.conclusion.negated)) for r in self.kb.rules}
        stratified = False
        for _ in range(len(head_keys) + 2):
            changed = False
            for r in self.kb.rules:
                needed = 0
                for c in r.conditions:
                    if c.negated:
                        needed = max(needed, level[(c.pred, False)] + 1, level[(c.pred, sign(True))])
                    else:
                        needed = max(needed, level[(c.pred, False)])
                key = (r.conclusion.pred, sign(r.conclusion.negated))
                if needed > level[key]:
                    level[key] = needed
                    changed = True
            if not changed:
                stratified = True
                break
        if not stratified:
            # negation inside a cycle: no clean layering exists, so fall back
            # to evaluating all rules together until nothing changes
            return [self.kb.rules]
        groups = defaultdict(list)
        for r in self.kb.rules:
            groups[level[(r.conclusion.pred, sign(r.conclusion.negated))]].append(r)
        return [groups[k] for k in sorted(groups)]

    def run(self, goal=None):
        """Forward chain to a fixed point. If a positive goal is given we stop
        as soon as it is derived (it can never be retracted afterwards)."""
        stop_early = goal is not None and not goal.negated
        if stop_early and goal in self.known:
            return self.known
        for rules in self._strata():
            # positive conditions first so variables get bound before NAF checks
            ordered = [Rule(tuple(sorted(r.conditions, key=lambda c: c.negated)), r.conclusion, r.text)
                       for r in rules]
            for _ in range(self.max_rounds):
                new_fact = False
                for rule in ordered:
                    for bindings, premises in list(self._match(list(rule.conditions), {}, [])):
                        head = substitute(rule.conclusion, bindings)
                        if any(is_variable(a) for a in head.args):
                            continue
                        if self._add(head, (rule, premises)):
                            new_fact = True
                            if stop_early and head == goal:
                                return self.known
                if not new_fact:
                    break
        return self.known

    def explain(self, fact, depth=0, seen=None):
        """Proof trace for a derived fact, as indented lines."""
        seen = seen or set()
        reason = self.derived_from.get(fact)
        pad = "  " * depth
        if reason is None:
            return [f"{pad}{literal_to_str(fact)}   [given fact]"]
        rule, premises = reason
        lines = [f"{pad}{literal_to_str(fact)}   [by rule: {rule.text}]"]
        for p in premises:
            if p in self.derived_from and p not in seen:
                lines += self.explain(p, depth + 1, seen | {fact})
            else:
                lines.append(f"{'  ' * (depth + 1)}{literal_to_str(p)}   [not derivable -> assumed false]")
        return lines


def answer_query(context, statement, closed_world=True, early_stop=True, split_by_sign=True):
    kb = KnowledgeBase.from_text(context)
    kind, items = parse_sentence(statement)
    if kind != "fact" or len(items) != 1:
        return None, kb, None
    query = items[0]
    engine = ForwardChainer(kb, closed_world=closed_world, split_by_sign=split_by_sign)
    engine.run(goal=query if early_stop else None)
    return engine.holds(query), kb, (engine, query)

## 4. Causal / counterfactual reasoning (CounterBench)
Every variable is a boolean function of its causes. The three classical steps are:
1. **Abduction**: enumerate the assignments of the root (exogenous) variables that agree with the evidence (`We observed ...`, and `X` is true in the real world when the question says *instead of X*).
2. **Intervention**: overwrite the variables named in the question and ignore their causes.
3. **Prediction**: propagate forward in topological order. The answer is *yes* only if the target is true in **every** consistent world.

In [8]:
import re
import itertools
from collections import namedtuple

Term = namedtuple("Term", ["var", "negated"])          # "not Blaf" -> Term('blaf', True)
Equation = namedtuple("Equation", ["kind", "terms"])  # kind: 'and' | 'or' | 'single'


def norm_var(word):
    return word.strip().strip(".?,").lower()


def parse_term(text):
    text = text.strip()
    if text.lower().startswith("not "):
        return Term(norm_var(text[4:]), True)
    return Term(norm_var(text), False)


def parse_cause_expression(text):
    """'A and B together' / 'A or not B' / 'not A' -> Equation"""
    text = re.sub(r"\btogether\b", "", text).strip()
    if " or " in text:
        return Equation("or", [parse_term(t) for t in text.split(" or ")])
    if " and " in text:
        return Equation("and", [parse_term(t) for t in text.split(" and ")])
    return Equation("single", [parse_term(text)])


def split_effects(text):
    return [t for t in re.split(r",\s*(?:and\s+)?|\s+and\s+", text.strip()) if t.strip()]


class CausalModel:
    def __init__(self):
        self.parents_eq = {}      # variable -> list of (Equation, effect_negated)
        self.variables = []
        self.priors = {}          # exogenous variable -> probability of being true
        self.observed = {}        # variable -> observed value in the real world
        self.statements = []      # readable form of the equations, for traces

    def _declare(self, var):
        if var not in self.variables:
            self.variables.append(var)

    @classmethod
    def from_text(cls, text):
        model = cls()
        text = re.sub(r"(?i)we know that", "", text)
        sentences = [s.strip() for s in re.split(r"\.\s*", text) if s.strip()]
        for sentence in sentences:
            low = sentence.lower()
            prior = re.match(r"^(\w+)\s*~\s*bern\(([\d.]+)\)$", low)
            if prior:
                model.priors[norm_var(prior.group(1))] = float(prior.group(2))
                continue
            if low.startswith("we observed") or low.startswith("it is observed"):
                observed_part = re.sub(r"^(we observed|it is observed( that)?)\s*", "", low)
                for item in split_effects(observed_part):
                    term = parse_term(item)
                    model._declare(term.var)
                    model.observed[term.var] = not term.negated
                continue
            model._parse_causal_sentence(sentence)
        return model

    def _parse_causal_sentence(self, sentence):
        # chunks separated by commas; a chunk without 'cause' continues the
        # effect list of the previous statement ("A causes B, C and D")
        current = None
        for chunk in re.split(r",\s*", sentence):
            chunk = re.sub(r"^and\s+", "", chunk.strip())
            m = re.match(r"^(.*?)\s+(?:causes|cause|has a direct effect on|have a direct effect on)\s+(.*)$", chunk)
            if m:
                current = parse_cause_expression(m.group(1))
                for term in current.terms:
                    self._declare(term.var)
                self._add_effects(current, m.group(2))
            elif current is not None:
                self._add_effects(current, chunk)

    def _add_effects(self, equation, effect_text):
        for effect in split_effects(effect_text):
            term = parse_term(effect)
            self._declare(term.var)
            self.parents_eq.setdefault(term.var, []).append((equation, term.negated))
            cause_str = f" {equation.kind} ".join(("not " if t.negated else "") + t.var for t in equation.terms)
            self.statements.append(f"{term.var} := {'not ' if term.negated else ''}({cause_str})")

    # ------------------------------------------------------------------
    def exogenous(self):
        return [v for v in self.variables if v not in self.parents_eq]

    def topological_order(self):
        order, done = [], set()
        def visit(v, stack=()):
            if v in done or v in stack:
                return
            for eq, _ in self.parents_eq.get(v, []):
                for t in eq.terms:
                    visit(t.var, stack + (v,))
            done.add(v)
            order.append(v)
        for v in self.variables:
            visit(v)
        return order

    @staticmethod
    def _evaluate(equation, values):
        bits = [values[t.var] != t.negated for t in equation.terms]
        return all(bits) if equation.kind == "and" else any(bits)

    def propagate(self, exogenous_values, interventions=None):
        """Compute every variable given exogenous values and forced values."""
        interventions = interventions or {}
        order = self.topological_order()
        values = {v: False for v in order}
        # one pass is enough for an acyclic graph; extra passes let a graph
        # with a feedback loop settle to a fixed point
        for _ in range(len(order) + 1):
            changed = False
            for var in order:
                if var in interventions:
                    new_value = interventions[var]
                elif var not in self.parents_eq:
                    new_value = exogenous_values[var]
                else:
                    # several equations for the same variable are combined with OR
                    new_value = any(self._evaluate(eq, values) != neg for eq, neg in self.parents_eq[var])
                if values[var] != new_value:
                    values[var] = new_value
                    changed = True
            if not changed:
                break
        return values

    def consistent_worlds(self, evidence):
        """Abduction: enumerate exogenous assignments that agree with evidence."""
        roots = self.exogenous()
        for combo in itertools.product([False, True], repeat=len(roots)):
            exo = dict(zip(roots, combo))
            world = self.propagate(exo)
            if all(world.get(v) == val for v, val in evidence.items()):
                weight = 1.0
                for r, val in exo.items():
                    p = self.priors.get(r, 0.5)
                    weight *= p if val else (1 - p)
                yield exo, world, weight


Question = namedtuple("Question", ["target", "interventions", "evidence", "kind"])


def parse_question(text):
    q = text.strip().rstrip("?").strip()
    # "Would Y occur if not X instead of X"
    m = re.match(r"^would (\w+) occur if (not )?(\w+) instead of (not )?(\w+)$", q, re.I)
    if m:
        x = norm_var(m.group(3))
        forced = not bool(m.group(2))
        return Question(norm_var(m.group(1)), {x: forced}, {x: not forced}, "counterfactual")
    # "Would Y occur if not X and not Z"
    m = re.match(r"^would (\w+) occur if (.+)$", q, re.I)
    if m:
        forced = {}
        for item in re.split(r"\s+and\s+|,\s*", m.group(2)):
            t = parse_term(item)
            forced[t.var] = not t.negated
        return Question(norm_var(m.group(1)), forced, {}, "joint")
    # "Assume not X, and based on this assumption, further suppose not Z. Would Y occur"
    m = re.match(r"^assume (.+?),? and based on this assumption, further suppose (.+?)\.\s*would (\w+) occur$", q, re.I)
    if m:
        first, second = parse_term(m.group(1)), parse_term(m.group(2))
        return Question(norm_var(m.group(3)),
                        {first.var: not first.negated, second.var: not second.negated}, {}, "nested")
    return None


def answer_counterfactual(context, question_text, nested_mode="both"):
    model = CausalModel.from_text(context)
    question = parse_question(question_text)
    if question is None or question.target not in model.variables:
        return None, model, question

    evidence = dict(model.observed)
    evidence.update(question.evidence)
    interventions = dict(question.interventions)

    worlds = list(model.consistent_worlds(evidence))
    if not worlds and question.evidence:
        # the factual premise implied by "instead of X" clashes with the
        # observations, so keep only what was explicitly observed
        worlds = list(model.consistent_worlds(dict(model.observed)))
    if not worlds and model.observed:
        # the observation cannot be produced by the model at all, so we read it
        # as a condition that is held fixed in the hypothetical world
        interventions = {**model.observed, **interventions}
        worlds = list(model.consistent_worlds({}))

    outcomes = []
    for exo, factual_world, weight in worlds:
        if question.kind == "nested":
            (x_var, x_val), (z_var, z_val) = list(question.interventions.items())
            if nested_mode == "outer_only":
                forced = {x_var: x_val}
            else:
                forced = {x_var: x_val, z_var: z_val}
            world = model.propagate(exo, {**interventions, **forced} if nested_mode != "outer_only" else forced)
        else:
            world = model.propagate(exo, interventions)
        outcomes.append((world[question.target], weight))

    if not outcomes:
        return None, model, question
    # entailed only if the target holds in every consistent world
    return all(v for v, _ in outcomes), model, question

## 5. First-order logic: CNF + resolution (FOLIO)
Formula → remove `→ ↔ ⊕` → push negations inward → rename variables and Skolemise `∃` → distribute `∨` over `∧` → clauses.
The query is negated and added; resolution with unification searches for the empty clause (set of support, shortest clause first, step and time limits).

In [9]:
import re
import time
import itertools

In [10]:
# ----------------------------------------------------------------------------
# Tokeniser and recursive-descent parser
# ----------------------------------------------------------------------------
SYMBOL_ALIASES = {"->": "→", "<->": "↔", "&": "∧", "|": "∨", "~": "¬", "!": "¬",
                  "⇒": "→", "⇔": "↔", "∨": "∨", "∧": "∧"}
TOKEN_RE = re.compile(r"\s*(<->|->|∀|∃|¬|∧|∨|→|↔|⊕|⇒|⇔|[()&|~!,]|[A-Za-z0-9_'.\-]+)")


def tokenize(text):
    tokens, pos = [], 0
    text = text.strip()
    while pos < len(text):
        m = TOKEN_RE.match(text, pos)
        if not m:
            pos += 1            # skip characters we do not understand
            continue
        tok = m.group(1)
        tokens.append(SYMBOL_ALIASES.get(tok, tok))
        pos = m.end()
    return tokens


class FormulaParser:
    """Precedence (weakest first): ↔, →, ⊕, ∨, ∧, then ¬ / quantifiers / atoms."""

    def __init__(self, text):
        self.tokens = tokenize(text)
        self.pos = 0
        self.bound = []          # stack of variables bound by quantifiers

    def peek(self):
        return self.tokens[self.pos] if self.pos < len(self.tokens) else None

    def take(self, expected=None):
        tok = self.peek()
        if expected is not None and tok != expected:
            raise ValueError(f"expected {expected} got {tok}")
        self.pos += 1
        return tok

    def parse(self):
        formula = self.parse_iff()
        if self.peek() is not None:
            raise ValueError(f"unexpected token {self.peek()}")
        return formula

    def parse_iff(self):
        left = self.parse_implies()
        while self.peek() == "↔":
            self.take()
            left = ("iff", left, self.parse_implies())
        return left

    def parse_implies(self):
        left = self.parse_xor()
        if self.peek() == "→":
            self.take()
            return ("imp", left, self.parse_implies())     # right associative
        return left

    def parse_xor(self):
        left = self.parse_or()
        while self.peek() == "⊕":
            self.take()
            left = ("xor", left, self.parse_or())
        return left

    def parse_or(self):
        left = self.parse_and()
        while self.peek() == "∨":
            self.take()
            left = ("or", left, self.parse_and())
        return left

    def parse_and(self):
        left = self.parse_unary()
        while self.peek() == "∧":
            self.take()
            left = ("and", left, self.parse_unary())
        return left

    def parse_unary(self):
        tok = self.peek()
        if tok == "¬":
            self.take()
            return ("not", self.parse_unary())
        if tok in ("∀", "∃"):
            self.take()
            var = self.take()
            self.bound.append(var)
            # the scope of a quantifier runs as far to the right as possible
            body = self.parse_iff() if self.peek() != "(" else self.parse_quantifier_body()
            self.bound.pop()
            return ("all" if tok == "∀" else "ex", var, body)
        if tok == "(":
            self.take()
            inner = self.parse_iff()
            self.take(")")
            return inner
        return self.parse_atom()

    def parse_quantifier_body(self):
        # "∀x (A(x) → B(x))" or "∀x (A(x)) → B(x)": keep reading after the brackets
        start = self.pos
        self.take("(")
        inner = self.parse_iff()
        self.take(")")
        if self.peek() in ("→", "↔", "⊕", "∨", "∧"):
            self.pos = start
            return self.parse_iff()
        return inner

    def parse_atom(self):
        name = self.take()
        if name is None or not re.match(r"[A-Za-z0-9_'.\-]+$", name):
            raise ValueError(f"bad atom {name}")
        args = []
        if self.peek() == "(":
            self.take()
            while self.peek() != ")":
                arg = self.take()
                if arg == ",":
                    continue
                if arg is None:
                    raise ValueError("unclosed argument list")
                args.append(self.make_term(arg))
            self.take(")")
        return ("atom", name.lower(), tuple(args))

    def make_term(self, name):
        if name in self.bound:
            return "?" + name
        if re.fullmatch(r"[a-z]", name):        # stray free variable -> treat as universal
            return "?" + name
        return name.lower()


def repair_formula(text):
    """Fix small annotation slips: '=' between terms and unbalanced brackets."""
    text = re.sub(r"([A-Za-z0-9_]+)\s*≠\s*([A-Za-z0-9_]+)", r"¬Equal(\1, \2)", text)
    text = re.sub(r"([A-Za-z0-9_]+)\s*=\s*([A-Za-z0-9_]+)", r"Equal(\1, \2)", text)
    depth, kept = 0, []
    for ch in text:
        if ch == ")":
            if depth == 0:
                continue                 # drop a closing bracket with no partner
            depth -= 1
        elif ch == "(":
            depth += 1
        kept.append(ch)
    return "".join(kept) + ")" * depth


def parse_formula(text):
    return FormulaParser(repair_formula(text)).parse()

In [11]:
# ----------------------------------------------------------------------------
# Conversion to clause normal form
# ----------------------------------------------------------------------------
def eliminate_connectives(f):
    op = f[0]
    if op == "atom":
        return f
    if op == "not":
        return ("not", eliminate_connectives(f[1]))
    if op in ("all", "ex"):
        return (op, f[1], eliminate_connectives(f[2]))
    a, b = eliminate_connectives(f[1]), eliminate_connectives(f[2])
    if op == "imp":
        return ("or", ("not", a), b)
    if op == "iff":
        return ("and", ("or", ("not", a), b), ("or", ("not", b), a))
    if op == "xor":
        return ("and", ("or", a, b), ("or", ("not", a), ("not", b)))
    return (op, a, b)


def push_negation(f, negate=False):
    op = f[0]
    if op == "atom":
        return ("not", f) if negate else f
    if op == "not":
        return push_negation(f[1], not negate)
    if op in ("all", "ex"):
        flipped = {"all": "ex", "ex": "all"}[op] if negate else op
        return (flipped, f[1], push_negation(f[2], negate))
    if op in ("and", "or"):
        flipped = {"and": "or", "or": "and"}[op] if negate else op
        return (flipped, push_negation(f[1], negate), push_negation(f[2], negate))
    raise ValueError(op)


class Skolemizer:
    counter = itertools.count()

    def __init__(self):
        pass

    def run(self, f, universals=(), renaming=None):
        renaming = renaming or {}
        op = f[0]
        if op == "atom":
            return ("atom", f[1], tuple(self.rename_term(t, renaming) for t in f[2]))
        if op == "not":
            return ("not", self.run(f[1], universals, renaming))
        if op == "all":
            fresh = f"?{f[1]}_{next(self.counter)}"
            return self.run(f[2], universals + (fresh,), {**renaming, "?" + f[1]: fresh})
        if op == "ex":
            n = next(self.counter)
            skolem = ("sk", f"sk{n}") + tuple(universals) if universals else f"sk{n}"
            return self.run(f[2], universals, {**renaming, "?" + f[1]: skolem})
        return (op, self.run(f[1], universals, renaming), self.run(f[2], universals, renaming))

    def rename_term(self, term, renaming):
        if isinstance(term, str) and term.startswith("?"):
            # free variables (no quantifier) stay universal
            return renaming.get(term, term)
        return term


MAX_CLAUSES_PER_FORMULA = 64


def to_clauses(f):
    """Distribute ∨ over ∧ and return a list of clauses (frozensets of literals)."""
    op = f[0]
    if op == "atom":
        return [frozenset([(False, f[1], f[2])])]
    if op == "not":
        a = f[1]
        return [frozenset([(True, a[1], a[2])])]
    if op == "and":
        return to_clauses(f[1]) + to_clauses(f[2])
    if op == "or":
        left, right = to_clauses(f[1]), to_clauses(f[2])
        if len(left) * len(right) > MAX_CLAUSES_PER_FORMULA:
            raise ValueError("CNF blow-up")
        return [a | b for a in left for b in right]
    raise ValueError(op)


def formula_to_clauses(text, negate=False):
    f = eliminate_connectives(parse_formula(text))
    f = push_negation(f, negate)
    f = Skolemizer().run(f)
    clauses = []
    for c in to_clauses(f):
        if not any((not neg, p, a) in c for neg, p, a in c):   # drop tautologies
            clauses.append(c)
    return clauses

In [12]:
# ----------------------------------------------------------------------------
# Unification and resolution
# ----------------------------------------------------------------------------
def is_var(t):
    return isinstance(t, str) and t.startswith("?")


def walk(t, subst):
    while is_var(t) and t in subst:
        t = subst[t]
    return t


def occurs(var, t, subst):
    t = walk(t, subst)
    if t == var:
        return True
    if isinstance(t, tuple):
        return any(occurs(var, a, subst) for a in t[2:])
    return False


def unify(a, b, subst):
    a, b = walk(a, subst), walk(b, subst)
    if a == b:
        return subst
    if is_var(a):
        return None if occurs(a, b, subst) else {**subst, a: b}
    if is_var(b):
        return None if occurs(b, a, subst) else {**subst, b: a}
    if isinstance(a, tuple) and isinstance(b, tuple) and a[1] == b[1] and len(a) == len(b):
        for x, y in zip(a[2:], b[2:]):
            subst = unify(x, y, subst)
            if subst is None:
                return None
        return subst
    return None


def apply_subst(t, subst):
    t = walk(t, subst)
    if isinstance(t, tuple):
        return t[:2] + tuple(apply_subst(a, subst) for a in t[2:])
    return t


def rename_clause(clause, suffix):
    def ren(t):
        if is_var(t):
            return t + suffix
        if isinstance(t, tuple):
            return t[:2] + tuple(ren(a) for a in t[2:])
        return t
    return frozenset((neg, p, tuple(ren(a) for a in args)) for neg, p, args in clause)


def resolve(c1, c2):
    """All resolvents of two clauses (c2 is renamed apart first)."""
    c2 = rename_clause(c2, "'")
    results = []
    for lit1 in c1:
        for lit2 in c2:
            if lit1[0] == lit2[0] or lit1[1] != lit2[1] or len(lit1[2]) != len(lit2[2]):
                continue
            subst = {}
            for x, y in zip(lit1[2], lit2[2]):
                subst = unify(x, y, subst)
                if subst is None:
                    break
            if subst is None:
                continue
            rest = (c1 - {lit1}) | (c2 - {lit2})
            new = frozenset((n, p, tuple(apply_subst(a, subst) for a in args)) for n, p, args in rest)
            if any((not n, p, a) in new for n, p, a in new):
                continue        # tautology
            results.append(new)
    return results


def canonical(clause):
    """Rename variables in a fixed order so duplicate clauses are detected."""
    mapping = {}
    def canon(t):
        if is_var(t):
            mapping.setdefault(t, f"?v{len(mapping)}")
            return mapping[t]
        if isinstance(t, tuple):
            return t[:2] + tuple(canon(a) for a in t[2:])
        return t
    lits = sorted(clause, key=lambda l: (l[1], l[0], str(l[2])))
    return frozenset((n, p, tuple(canon(a) for a in args)) for n, p, args in lits)


def resolution_prove(kb_clauses, goal_clauses, max_steps=400, max_clauses=4000, time_limit=1.0):
    """Set-of-support resolution: only resolve with clauses that descend from
    the negated goal. Returns (proved, proof_steps)."""
    start = time.perf_counter()
    usable = list(kb_clauses)
    seen = {canonical(c) for c in usable}
    support = []
    for c in goal_clauses:
        if canonical(c) not in seen:
            seen.add(canonical(c))
            support.append(c)
    parents = {c: None for c in support}
    processed = []
    steps = 0
    while support and steps < max_steps and len(seen) < max_clauses:
        if time.perf_counter() - start > time_limit:
            break
        support.sort(key=len)                  # shortest clause first (unit preference)
        given = support.pop(0)
        steps += 1
        for other in usable + processed:
            for resolvent in resolve(given, other):
                if not resolvent:
                    parents[resolvent] = (given, other)
                    return True, (resolvent, parents)
                key = canonical(resolvent)
                if key in seen:
                    continue
                seen.add(key)
                parents[resolvent] = (given, other)
                support.append(resolvent)
        processed.append(given)
    return False, None


def clause_to_str(clause):
    if not clause:
        return "□ (empty clause)"
    def term(t):
        if isinstance(t, tuple):
            return f"{t[1]}({', '.join(term(a) for a in t[2:])})"
        return t.lstrip("?")
    return " ∨ ".join(("¬" if n else "") + f"{p}({', '.join(term(a) for a in args)})" for n, p, args in clause)


def prove_fol(premises_fol, conclusion_fol, **limits):
    """premises_fol: list of formula strings. Returns True / False / None (parse failure)."""
    kb = []
    for line in premises_fol:
        if not line.strip():
            continue
        try:
            kb.extend(formula_to_clauses(line))
        except Exception:
            continue                 # skip a premise we cannot read
    try:
        goal = formula_to_clauses(conclusion_fol, negate=True)
    except Exception:
        return None, None
    return resolution_prove(kb, goal, **limits)


def explain_refutation(proof):
    """Turn the parent links of a refutation into readable resolution steps."""
    empty, parents = proof
    lines, done = [], set()
    def visit(clause):
        if clause in done or parents.get(clause) is None:
            return
        done.add(clause)
        left, right = parents[clause]
        visit(left)
        visit(right)
        lines.append(f"resolve [{clause_to_str(left)}] with [{clause_to_str(right)}]  =>  {clause_to_str(clause)}")
    visit(empty)
    return lines

## 5b. English → FOL translation (no annotation needed)
For free-form English such as FOLIO premises we translate the sentence patterns we can recognise into FOL strings
(`All A are B`, `No A is B`, `Some A are B`, `If ..., then ...`, `X is A or B`, `X is neither A nor B`, `A who B are C`,
named individuals) and send them to the same resolution prover. Sentences outside this grammar are skipped, so the
answer is *True* only if the query follows from the sentences that were understood; if the translated premises
also prove the opposite, the result is not trusted.

In [13]:
import re

ARTICLES = {"a", "an", "the", "some", "any"}
IRREGULAR = {"has": "have", "does": "do", "is": "be", "are": "be", "was": "be", "were": "be",
             "people": "person", "children": "child", "men": "man", "women": "woman", "mice": "mouse"}
NEGATIONS = [("is not ", ""), ("are not ", ""), ("isn't ", ""), ("aren't ", ""), ("was not ", ""),
             ("does not ", ""), ("do not ", ""), ("doesn't ", ""), ("don't ", ""), ("did not ", ""),
             ("cannot ", "can "), ("can not ", "can "), ("can't ", "can "), ("will not ", "will "),
             ("won't ", "will "), ("never ", "")]
QUANTIFIER_RE = re.compile(r"^(?:all|every|each|any|no)\b(?: of)?(?: the)?\s+(.*)$")
REFERRING_PRONOUNS = {"he", "she", "they", "it", "him", "her", "them"}
GROUP_SUBJECTS = {"everyone", "everybody", "anyone", "anybody", "someone", "somebody", "something", "everything",
                  "anything", "person", "people", "thing", "things", "you", "they", "it", "he", "she"}


def singular(word):
    if word in IRREGULAR:
        return IRREGULAR[word]
    if len(word) > 4 and word.endswith("ies"):
        return word[:-3] + "y"
    if len(word) > 4 and word.endswith(("sses", "ches", "shes", "xes")):
        return word[:-2]
    if len(word) > 3 and word.endswith("s") and not word.endswith(("ss", "us", "is")):
        return word[:-1]
    return word


def predicate_name(phrase):
    words = re.findall(r"[a-z0-9]+", phrase.lower().replace("'s", ""))
    words = [singular(w) for w in words if w not in ARTICLES]
    return "_".join(words) if words else None


def constant_name(phrase):
    words = re.findall(r"[a-z0-9]+", phrase.lower())
    words = [w for w in words if w not in ARTICLES]
    return "c_" + "_".join(words) if words else None


GROUP_TAIL = re.compile(r"\s+(?:things?|people|persons?|ones?|individuals?|beings?)$", re.I)


def property_formula(phrase, var):
    """'a flightless bird' / 'not kind' / 'either red or big' -> FOL string about var."""
    phrase = phrase.strip().strip(",.").strip()
    phrase = re.sub(r"^(?:he|she|they|it|who|that)\s+", "", phrase, flags=re.I)
    phrase = re.sub(r"^(?:either|both)\s+", "", phrase)
    if not phrase:
        return None
    m = re.match(r"^(?:is |are )?neither (.+?) nor (.+)$", phrase, re.I)
    if m:
        a, b = property_formula(m.group(1), var), property_formula(m.group(2), var)
        return f"(¬{a} ∧ ¬{b})" if a and b else None
    if " or " in phrase:
        parts = [property_formula(p, var) for p in re.split(r",?\s+or\s+|,\s*", phrase) if p.strip()]
        return None if None in parts else "(" + " ∨ ".join(parts) + ")"
    if " and " in phrase:
        parts = [property_formula(p, var) for p in re.split(r",?\s+and\s+|,\s*", phrase) if p.strip()]
        return None if None in parts else "(" + " ∧ ".join(parts) + ")"
    negated = False
    low = phrase.lower()
    for neg, repl in NEGATIONS + [("not ", ""), ("no longer ", "")]:
        if low.startswith(neg):
            negated, low = True, repl + low[len(neg):]
            break
    low = re.sub(r"^(?:is|are|be|am|was|were)\s+", "", low)
    name = predicate_name(low)
    if name is None:
        return None
    atom = f"{name}({var})"
    words = name.split("_")
    if 1 < len(words) <= 4 and not any(w in ("of", "in", "at", "to", "for", "with", "by", "on") for w in words):
        atom = f"({atom} ∧ {words[-1]}({var}))"      # "flightless bird" also gives "bird"
    return f"¬{atom}" if negated else atom


VERB_START = re.compile(r"\s(?=(?:is|are|was|were|has|have|had|does|do|did|can|cannot|will|won't|"
                        r"doesn't|don't|isn't|aren't|never|either|neither)\b)")


def split_subject_verb(text):
    """Split 'X <verb phrase>' at the first verb-ish word after the subject.
    With a relative clause ('A who are B are C') the clause keeps its own verb."""
    rel = re.search(r"\s(?:who|that|which)\s", text)
    if rel:
        after = text[rel.end():]
        if "," in after:
            head, tail = text.rsplit(",", 1)
            return head.strip(), tail.strip()
        cuts = [m.start() for m in VERB_START.finditer(" " + after)]
        if len(cuts) >= 2:
            k = rel.end() + cuts[1] - 1
            return text[:k].strip(), text[k:].strip()
    m = re.match(r"^(.+?)\s+((?:is|are|was|were|has|have|had|does|do|did|can|cannot|will|won't|"
                 r"doesn't|don't|isn't|aren't|never|either|neither)\b.*)$", text)
    if m:
        return m.group(1), m.group(2)
    words = text.split()
    for i in range(1, len(words)):
        if words[i].endswith(("s", "ed")) and not words[i][0].isupper() and words[i] not in ("its", "this"):
            return " ".join(words[:i]), " ".join(words[i:])
    return None, None


def is_name(subject):
    words = subject.split()
    return bool(words) and all(w[0].isupper() or w[0].isdigit() or w in ("of", "the", "de", "von") for w in words) \
        and words[0].lower() not in GROUP_SUBJECTS


def restrictor_formula(text, var):
    """'Harvard students who are diligent' -> harvard_student(x) ∧ diligent(x)"""
    parts = re.split(r"\s+(?:who|that|which|whom)\s+", text.strip().strip(","), maxsplit=1)
    noun = GROUP_TAIL.sub("", parts[0].strip())
    noun = re.sub(r"^(?:someone|somebody|something|anyone|anybody|everyone|everybody|people|persons?|things?)\s+",
                  "", noun, flags=re.I)
    pieces = []
    if noun and noun.lower() not in GROUP_SUBJECTS:
        p = property_formula(noun, var)
        if p is None:
            return None
        pieces.append(p)
    if len(parts) > 1:
        p = property_formula(parts[1], var)
        if p is None:
            return None
        pieces.append(p)
    return " ∧ ".join(pieces)


def guarded(restriction, body, quantifier="all"):
    if restriction is None or body is None:
        return None
    if quantifier == "all":
        return f"∀x ({restriction} → {body})" if restriction else f"∀x ({body})"
    return f"∃x ({restriction} ∧ {body})" if restriction else f"∃x ({body})"


def translate_sentence(sentence):
    """Return an FOL string for one English sentence, or None if unsupported."""
    s = sentence.strip().rstrip(".").strip()
    s = re.sub(r"\s+", " ", s)
    if not s:
        return None
    low = s.lower()

    # If ..., then ...   (one variable, or about named individuals)
    m = re.match(r"^if (.+?),? then (.+)$", s, re.I) or re.match(r"^if ([^,]+), (.+)$", s, re.I)
    if m:
        cond, concl = m.group(1), m.group(2)
        subj_c, vp_c = split_subject_verb(cond)
        subj_h, vp_h = split_subject_verb(concl)
        if not (vp_c and vp_h):
            return None
        if is_name(subj_c):
            const_c = constant_name(subj_c)
            # the conclusion keeps its own subject: "If Alice ..., then Bob ..."
            if subj_h.lower() in REFERRING_PRONOUNS:
                const_h = const_c
            elif is_name(subj_h) or subj_h.lower().startswith("the "):
                const_h = constant_name(subj_h)
            else:
                return None
            a, b = property_formula(vp_c, const_c), property_formula(vp_h, const_h)
            return f"{a} → {b}" if a and b else None
        r = restrictor_formula(re.sub(r"^(?:a|an|the)\s+", "", subj_c, flags=re.I), "x")
        # "If someone is X then Bob is Y": the conclusion is about Bob, not about x
        head_term = constant_name(subj_h) if is_name(subj_h) else "x"
        a, b = property_formula(vp_c, "x"), property_formula(vp_h, head_term)
        if r is None or not (a and b):
            return None
        return guarded(f"{r} ∧ {a}" if r else a, b)

    # No A is B
    m = re.match(r"^no(?:ne of the| one)? (.+)$", low)
    if m:
        subj, vp = split_subject_verb(s[len(s) - len(m.group(1)):])
        if not vp and len(m.group(1).split()) > 1:
            subj, vp = m.group(1).split(" ", 1)
        if vp:
            b = property_formula(vp, "x")
            return guarded(restrictor_formula(subj, "x"), f"¬({b})" if b else None)

    # Some A are B / There is a A who B
    m = re.match(r"^(?:some|there (?:is|are|exists?) (?:a|an|some|at least one))\s+(.+)$", low)
    if m:
        rest = s[len(s) - len(m.group(1)):]
        if re.match(r"^there", low):
            r = restrictor_formula(rest, "x")
            return f"∃x ({r})" if r else None
        subj, vp = split_subject_verb(rest)
        if vp:
            return guarded(restrictor_formula(subj, "x"), property_formula(vp, "x"), "some")
        return None

    # All / every / each A (who B) are C
    m = QUANTIFIER_RE.match(low)
    if m or low.split()[0] in ("everyone", "everybody", "anyone", "anybody"):
        rest = s[len(s) - len(m.group(1)):] if m else s
        subj, vp = split_subject_verb(rest)
        if vp:
            return guarded(restrictor_formula(subj, "x"), property_formula(vp, "x"))
        return None

    subj, vp = split_subject_verb(s)
    if not vp:
        return None
    plural_verb = re.match(r"^(?:are|aren't|do|don't|have)\b", vp.lower())
    # Generic plural: "Penguins are flightless birds", "People who X are Y"
    if plural_verb and (not is_name(subj) or subj.split()[-1].lower().endswith("s")):
        return guarded(restrictor_formula(subj, "x"), property_formula(vp, "x"))
    # Named individual: "Ralph is a German writer", "The Bronx is in New York"
    if is_name(subj) or low.startswith("the "):
        return property_formula(vp, constant_name(subj))
    return None


def prove_from_english(context, statement, time_limit=0.3):
    """English-only route: translate what we can, then run resolution.
    Returns (answer or None, number of premises translated)."""
    premises = [translate_sentence(p) for p in re.split(r"\n+|(?<=[.!?])\s+(?=[A-Z])", context)]
    premises = [p for p in premises if p]
    goal = translate_sentence(statement)
    if not premises or not goal:
        return None, len(premises)
    proved, proof = prove_fol(premises, goal, max_steps=300, time_limit=time_limit)
    if not proved:
        return False, len(premises)
    # a KB that also proves the opposite is inconsistent after translation: do not trust it
    refuted, _ = prove_fol(premises, f"¬({goal})", max_steps=300, time_limit=time_limit)
    return (not refuted), len(premises)

## 6. Putting it together
`reason()` picks the module from the shape of the input: causal question → causal engine; FOL given → resolution; fully parsed rule text → forward chaining; other English → English→FOL + resolution; otherwise *False* (not entailed).

In [14]:
CAUSAL_QUESTION = re.compile(r"^\s*(would \w+ occur|assume .* would \w+ occur)", re.I | re.S)


def has_value(x):
    return isinstance(x, str) and x.strip() != ""


def as_text(value):
    """Inputs may arrive as a string, a list of sentences, or be missing."""
    if isinstance(value, (list, tuple)):
        return "\n".join(str(v).strip() for v in value if str(v).strip())
    if value is None or (isinstance(value, float) and value != value):   # None or NaN
        return ""
    return str(value)


def reason(context, statement, context_fol=None, statement_fol=None):
    """Main entry point. Returns (prediction, module_used, trace_lines)."""
    context, statement = as_text(context), as_text(statement)
    context_fol, statement_fol = as_text(context_fol), as_text(statement_fol)
    if not context.strip() or not statement.strip():
        return False, "default", None
    # 1) causal / counterfactual questions
    if CAUSAL_QUESTION.match(statement) and re.search(r"\bcause", context, re.I):
        answer, model, question = answer_counterfactual(context, statement)
        if answer is not None:
            return answer, "causal", None

    # 2) first-order logic annotation available -> resolution prover
    if has_value(context_fol) and has_value(statement_fol):
        proved, proof = prove_fol(context_fol.split("\n"), statement_fol,
                                  max_steps=300, time_limit=0.3)
        if proved is not None:
            return bool(proved), "resolution", proof

    # 3) natural language rules -> parser + stratified forward chaining.
    #    Only trusted when (almost) every sentence of the context was understood,
    #    because the closed-world assumption treats anything unparsed as false.
    answer, kb, extra = answer_query(context, statement, closed_world=CLOSED_WORLD)
    if answer is not None and kb.parsed_fraction() >= 0.9:
        return answer, "forward-chaining", extra

    # 4) free-form English -> our own FOL translation -> resolution (open world)
    answer, translated = prove_from_english(context, statement)
    if answer is not None:
        return answer, "english-to-fol", None

    # 5) nothing usable: fall back to "not entailed"
    return False, "default", None


def run_on_dataframe(df):
    """Predict every row, return predictions, modules used and total time."""
    predictions, modules = [], []
    has_fol = "context_fol" in df.columns
    start = time.perf_counter()
    for row in df.itertuples(index=False):
        pred, module, _ = reason(row.context, row.statement,
                                 row.context_fol if has_fol else None,
                                 row.statement_fol if has_fol else None)
        predictions.append(pred)
        modules.append(module)
    elapsed = time.perf_counter() - start
    return predictions, modules, elapsed

## 7. Evaluation on the public datasets (accuracy and inference time)

In [15]:
if PUBLIC_DATA_AVAILABLE:
    results = {}
    overall_correct, overall_total, overall_time = 0, 0, 0.0

    for name, df in [("RuleTaker (test)", ruletaker_test),
                     ("FOLIO (train)", folio_train),
                     ("FOLIO (validation)", folio_val),
                     ("CounterBench", counterbench)]:
        preds, modules, seconds = run_on_dataframe(df)
        df = df.assign(pred=preds, module=modules)
        df["correct"] = df["pred"] == df["label"]
        results[name] = df
        overall_correct += df["correct"].sum()
        overall_total += len(df)
        overall_time += seconds
        print(f"{name:20s} accuracy = {100 * df['correct'].mean():6.2f}%   "
              f"time = {seconds:7.2f}s   ({1000 * seconds / len(df):.3f} ms/query)   "
              f"modules = {dict(Counter(modules))}")

    print(f"\nOVERALL accuracy = {100 * overall_correct / overall_total:.2f}%  "
          f"on {overall_total} queries,  total inference time = {overall_time:.2f}s")
else:
    print("skipped: public datasets not available")

RuleTaker (test)     accuracy =  99.97%   time =  132.60s   (0.873 ms/query)   modules = {'forward-chaining': 151911}
FOLIO (train)        accuracy =  81.02%   time =   26.10s   (26.076 ms/query)   modules = {'resolution': 999, 'english-to-fol': 2}
FOLIO (validation)   accuracy =  84.73%   time =    7.82s   (38.522 ms/query)   modules = {'resolution': 203}
CounterBench         accuracy =  94.58%   time =    0.48s   (0.403 ms/query)   modules = {'causal': 1200}

OVERALL accuracy = 99.79%  on 154315 queries,  total inference time = 167.00s


In [16]:
if PUBLIC_DATA_AVAILABLE:
    # accuracy per reasoning depth / question type
    for name in ["RuleTaker (test)", "CounterBench", "FOLIO (validation)"]:
        df = results[name]
        table = df.groupby("subset")["correct"].agg(["mean", "count"])
        table["mean"] = (100 * table["mean"]).round(2)
        print(f"--- {name} ---")
        print(table.rename(columns={"mean": "accuracy %", "count": "n"}).to_string(), "\n")
else:
    print("skipped: public datasets not available")

--- RuleTaker (test) ---
                    accuracy %      n
subset                               
NatLang                 100.00   7793
depth-0                 100.00  17347
depth-1                  99.99  17788
depth-2                  99.99  17741
depth-3                  99.96  17982
depth-3ext               99.96  23222
depth-3ext-NatLang       99.97  31017
depth-5                  99.92  19021 

--- CounterBench ---
             accuracy %    n
subset                      
backdoor           87.5  200
basic             100.0  250
conditional        92.0  250
joint              96.0  250
nested             96.0  250 

--- FOLIO (validation) ---
           accuracy %   n
subset                   
False           95.16  62
True            62.50  72
Uncertain       98.55  69 



## 8. Experiments
Closed vs open world, stratification by (predicate, sign) vs predicate only, early stopping, parser coverage, CounterBench reading of nested questions, and FOLIO answered three ways: always-False baseline, English only (our translation), gold FOL annotation.

In [17]:
if PUBLIC_DATA_AVAILABLE:
    # Experiments on a fixed random sample of RuleTaker (10,000 queries)
    sample = ruletaker_test.sample(n=min(10000, len(ruletaker_test)), random_state=0)


    def evaluate_rule_engine(df, closed_world=True, early_stop=True, split_by_sign=True):
        correct, start = 0, time.perf_counter()
        for row in df.itertuples(index=False):
            pred, _, _ = answer_query(row.context, row.statement, closed_world=closed_world,
                                      early_stop=early_stop, split_by_sign=split_by_sign)
            correct += (bool(pred) == row.label)
        return 100 * correct / len(df), time.perf_counter() - start


    for title, kwargs in [("closed world + early stop (final)", dict(closed_world=True, early_stop=True)),
                          ("strata by predicate name only",      dict(closed_world=True, split_by_sign=False)),
                          ("closed world, full closure",        dict(closed_world=True, early_stop=False)),
                          ("open world (no negation as failure)", dict(closed_world=False, early_stop=True))]:
        acc, secs = evaluate_rule_engine(sample, **kwargs)
        print(f"{title:38s} accuracy = {acc:6.2f}%   time = {secs:6.2f}s")

    # parser coverage
    unparsed = sum(len(KnowledgeBase.from_text(c).unparsed) for c in sample["context"])
    total = sum(len(split_sentences(c)) for c in sample["context"])
    print(f"\nparser coverage on this 10,000-query sample: {100 * (1 - unparsed / total):.2f}% of {total} sentences")

    # CounterBench: how nested questions are read
    for mode in ["both", "outer_only"]:
        sub = counterbench[counterbench["subset"] == "nested"]
        acc = sum(bool(answer_counterfactual(r.context, r.statement, nested_mode=mode)[0]) == r.label
                  for r in sub.itertuples(index=False)) / len(sub)
        print(f"CounterBench nested, intervene on {mode:10s}: {100 * acc:.2f}%")

    # FOLIO: three ways of answering the same questions
    def folio_english_only(df):
        """Hide the FOL annotation, so only the English text can be used."""
        preds, modules, seconds = run_on_dataframe(df.drop(columns=["context_fol", "statement_fol"]))
        acc = 100 * (pd.Series(preds, index=df.index) == df["label"]).mean()
        return acc, Counter(modules), seconds

    for name, df in [("validation", folio_val), ("train", folio_train)]:
        base = 100 * (1 - df["label"].mean())
        with_fol = 100 * results[f"FOLIO ({name})"]["correct"].mean()
        english, modules, secs = folio_english_only(df)
        print(f"\nFOLIO {name:10s}: always-False {base:.2f}% | English only (our translation) {english:.2f}% "
              f"| gold FOL annotation {with_fol:.2f}%")
        print(f"   English-only route used: {dict(modules)}  ({secs:.1f}s)")
else:
    print("skipped: public datasets not available")

closed world + early stop (final)      accuracy =  99.94%   time =   9.05s
strata by predicate name only          accuracy =  99.92%   time =   8.65s
closed world, full closure             accuracy =  99.94%   time =   9.21s
open world (no negation as failure)    accuracy =  75.11%   time =   8.19s

parser coverage on this 10,000-query sample: 100.00% of 146351 sentences
CounterBench nested, intervene on both      : 96.00%
CounterBench nested, intervene on outer_only: 78.00%

FOLIO validation: always-False 64.53% | English only (our translation) 66.50% | gold FOL annotation 84.73%
   English-only route used: {'english-to-fol': 167, 'default': 29, 'forward-chaining': 7}  (3.2s)

FOLIO train     : always-False 61.24% | English only (our translation) 63.34% | gold FOL annotation 81.02%
   English-only route used: {'english-to-fol': 837, 'default': 155, 'forward-chaining': 9}  (11.5s)


## 9. Reasoning traces

In [18]:
def show_trace(context, statement, context_fol=None, statement_fol=None):
    pred, module, extra = reason(context, statement, context_fol, statement_fol)
    print(f"Query: {statement}\nModule: {module}   Prediction: {pred}")
    if module == "forward-chaining":
        engine, query = extra
        target = query if not query.negated else Literal(query.pred, query.args, False)
        if target in engine.known:
            print("\n".join(engine.explain(target)))
        else:
            print(f"{literal_to_str(target)} cannot be derived (closed world -> false)")
    elif module == "resolution" and extra:
        print("\n".join(explain_refutation(extra)))
    elif module == "english-to-fol":
        premises = [translate_sentence(p) for p in re.split(r"\n+|(?<=[.!?])\s+(?=[A-Z])", context)]
        premises = [p for p in premises if p]
        goal = translate_sentence(statement)
        print("translated premises:", *premises, sep="\n   ")
        print("translated query:   ", goal)
        proved, proof = prove_fol(premises, goal)
        if proved:
            print("\n".join(explain_refutation(proof)))
    elif module == "causal":
        model = CausalModel.from_text(context)
        question = parse_question(statement)
        print("structural equations:", "; ".join(model.statements))
        print("observed:", model.observed, "| intervention:", question.interventions,
              "| real-world evidence:", question.evidence)
        evidence = {**model.observed, **question.evidence}
        for exo, real_world, _ in model.consistent_worlds(evidence):
            hypothetical = model.propagate(exo, question.interventions)
            print(f"  consistent root values {exo} -> after intervention "
                  f"{question.target} = {hypothetical[question.target]}")
    print("-" * 80)


show_trace("Bob is young. Bob is kind. If someone is young then they are smart. "
           "If someone is smart and kind then they are successful.", "Bob is successful.")

if PUBLIC_DATA_AVAILABLE:
    deep_pool = ruletaker_test[(ruletaker_test["subset"] == "depth-5") & ruletaker_test["label"]
                               & ~ruletaker_test["statement"].str.contains(" not ")]
    deep = deep_pool.iloc[min(3, len(deep_pool) - 1)]
    show_trace(deep.context, deep.statement)

    cb = counterbench[counterbench["subset"] == "conditional"].iloc[0]
    show_trace(cb.context, cb.statement)

    # shortest FOLIO story that the prover proves correctly
    fv = results["FOLIO (validation)"]
    fo = fv[fv["label"] & fv["correct"]].sort_values("context", key=lambda s: s.str.len()).iloc[0]
    print(fo.context, "\n")
    print("FOL premises:\n" + fo.context_fol, "\nFOL query:", fo.statement_fol, "\n")
    show_trace(fo.context, fo.statement, fo.context_fol, fo.statement_fol)

    # the same FOLIO story answered from the English text only (annotation hidden)
    show_trace(fo.context, fo.statement)

Query: Bob is successful.
Module: forward-chaining   Prediction: True
successful(bob)   [by rule: If someone is smart and kind then they are successful.]
  smart(bob)   [by rule: If someone is young then they are smart.]
    young(bob)   [given fact]
  kind(bob)   [given fact]
--------------------------------------------------------------------------------
Query: The cat is kind.
Module: forward-chaining   Prediction: True
kind(cat)   [by rule: If something sees the lion and it is not blue then it is kind.]
  see(cat, lion)   [by rule: If something chases the cat then the cat sees the lion.]
    chase(mouse, cat)   [by rule: If something is kind then it chases the cat.]
      kind(mouse)   [by rule: If something sees the lion and it is not blue then it is kind.]
        see(mouse, lion)   [given fact]
        ¬blue(mouse)   [not derivable -> assumed false]
  ¬blue(cat)   [not derivable -> assumed false]
--------------------------------------------------------------------------------
Qu

## 10. submission.csv
Set `TEST_FILE` in the setup cell to the private test file and run all cells. The file gets columns `id,label`. If the file has a label column the final accuracy is printed as well.

In [19]:
COLUMN_NAMES = {
    "context": ["context", "premises", "given_info", "kb", "knowledge_base", "theory"],
    "statement": ["statement", "question", "conclusion", "query", "hypothesis"],
    "label": ["label", "flag", "answer", "gold"],
    "context_fol": ["context_fol", "premises-FOL", "premises_fol"],
    "statement_fol": ["statement_fol", "conclusion-FOL", "conclusion_fol"],
}


def to_bool(value):
    return str(value).strip().lower() in ("true", "yes", "entailment", "1", "t")


def load_any_file(path):
    if path.endswith(".csv"):
        df = pd.read_csv(path)
    elif path.endswith(".jsonl"):
        df = pd.DataFrame(read_jsonl(path))
    else:
        df = pd.read_json(path)
    out = pd.DataFrame()
    out["id"] = df["id"] if "id" in df.columns else range(len(df))
    for target, options in COLUMN_NAMES.items():
        found = next((c for c in options if c in df.columns), None)
        if found is not None:
            # premises may be given as a list of sentences: turn them into one text
            out[target] = df[found].map(to_bool) if target == "label" else df[found].map(as_text)
    missing = [t for t in ("context", "statement") if t not in out.columns]
    if missing:
        raise ValueError(f"{path}: no column found for {missing}. Accepted names: "
                         + "; ".join(f"{t} = {COLUMN_NAMES[t]}" for t in missing))
    return out


def predict_file(path=None, output=SUBMISSION_PATH):
    if path is None:
        # no private file given: use every public evaluation example
        if not PUBLIC_DATA_AVAILABLE:
            raise FileNotFoundError("Set TEST_FILE to the test file (public data not found either).")
        test_df = pd.concat([ruletaker_test, folio_val, counterbench], ignore_index=True)
    else:
        test_df = load_any_file(path)

    preds, modules, seconds = run_on_dataframe(test_df)
    pd.DataFrame({"id": test_df["id"], "label": preds}).to_csv(output, index=False)

    print(f"wrote {len(preds)} predictions to {output}")
    if "label" in test_df.columns:
        accuracy = 100 * (pd.Series(preds) == test_df["label"].reset_index(drop=True)).mean()
        print(f"Final accuracy: {accuracy:.2f}%")
    print(f"Total inference time: {seconds:.2f}s  ({1000 * seconds / len(preds):.3f} ms per query)")


predict_file(TEST_FILE)

wrote 153314 predictions to submission.csv
Final accuracy: 99.91%
Total inference time: 141.27s  (0.921 ms per query)
